## 2. Reward Model

In this section, we train a **reward model** to evaluate the quality or relevance of generated content.  
We skip the earlier stages of the LLM pipeline — **(1) pretraining** and **(2) supervised fine-tuning** — and start from an already **instruction-tuned model**.

---

### Objective

The goal is to learn a reward function  
$$
r_\phi(x, y)
$$  
that assigns a scalar value to a model output $y$ given an input $x$.  

Here, $x \sim \mathcal{D}$ represents a sample drawn from the data distribution,  and $ y \sim \pi_\theta(y \mid x) $ is a response generated by the language model.
This value represents how *preferred* or *relevant* the output is, acting as a proxy for human feedback.


### Approach

A common method is to frame reward modeling as a **regression task**, where the model predicts an **unbounded scalar reward**. 
 
We use a pretrained language model $\pi_\theta(y \mid x)$ and attach a final linear layer with a single neuron, which will be trained to output the predicted reward value $r_\phi(x, y)$ from the positive and negative prompts in the prompt database $\mathcal{D}$.


### Intuition

The reward model learns to **score** responses so that higher rewards correspond to outputs that align better with human preferences.  
This model will later guide reinforcement learning updates (e.g., in PPO) to fine-tune the policy model.

In [ ]:
import transformers
from transformers import AutoModelForSequenceClassification
from trl import RewardConfig, RewardTrainer
from peft import LoraConfig 
import pandas as pd 
import torch
from datasets import load_dataset, DatasetDict
from huggingface_hub import login
import os

### Training the Reward Model

The reward model $r_\phi(x, y)$ is trained to predict human (or synthetic) preferences over pairs of model outputs.

---

#### Preference-Based Objective

Given a prompt $x$ and two candidate responses $(y^+, y^-)$,  
where $y^+$ is preferred over $y^-$ according to human feedback,  
the model should assign a higher reward to $y^+$:

$$
r_\phi(x, y^+) > r_\phi(x, y^-)
$$

To enforce this, we use a **pairwise logistic loss** (used in RLHF, e.g. in InstructGPT):

$$
\mathcal{L}_{\text{RM}}(\phi)
= - \mathbb{E}_{(x, y^+, y^-) \sim \mathcal{D}}
  \left[
    \log \sigma\!\left(r_\phi(x, y^+) - r_\phi(x, y^-)\right)
  \right]
$$

where $\sigma(\cdot)$ is the sigmoid function:
$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

This encourages the model to assign a higher score to the preferred completion.

---

#### Intuition

- If $r_\phi(x, y^+) \gg r_\phi(x, y^-)$,  
  then $\sigma(r_\phi(x, y^+) - r_\phi(x, y^-)) \approx 1$,  
  and the loss is small.  
- If the model ranks them incorrectly, the loss is large.  

---

#### Alternative (Regression) Objective

If explicit preference pairs are unavailable,  
the reward model can also be trained via regression to approximate scalar feedback values:

$$
\mathcal{L}_{\text{reg}}(\phi)
= \mathbb{E}_{(x, y, R) \sim \mathcal{D}}
  \left[ (r_\phi(x, y) - R)^2 \right]
$$

---

We will use pairwise as the training objective rather than the regression training loss.



In [ ]:
SEED = 42
SHUFFLE_SEED = 42
HF_DATASET_ID = "eZWALT/rlhf_reward_data_raw"  
HUB_REPO_ID = "eZWALT/rlhf_reward_splits_raw"  
PUSH_TO_HUB = True


SEED = 42
SHUFFLE_SEED = 42
ds = load_dataset(HF_DATASET_ID, split="train")   


# 2) shuffle then split to 80/10/10
# First shuffle the entire dataset (important to get a random split)
ds = ds.shuffle(seed=SHUFFLE_SEED)

# Split 80/20 (train / rest)
train_test = ds.train_test_split(test_size=0.20, seed=SEED)
train_ds = train_test["train"]          # ~80%
rest_ds = train_test["test"]            # ~20%

# Split the rest into half/half -> validation/test = 10% each
val_test = rest_ds.train_test_split(test_size=0.5, seed=SEED)
val_ds = val_test["train"]              # ~10%
test_ds = val_test["test"]              # ~10%

# Put into DatasetDict
dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

print(dataset_dict)
print("Train / Val / Test sizes:", len(dataset_dict["train"]), len(dataset_dict["validation"]), len(dataset_dict["test"]))

# 3a) Save locally for later use (optional)
dataset_dict.save_to_disk("../data/./hf_rlhf_splits")

# 3b) Push the new split dataset to the Hub (optional)
if PUSH_TO_HUB:
    dataset_dict.push_to_hub(HUB_REPO_ID, private=True)
    print("Pushed split dataset to hub at:", HUB_REPO_ID)


Using the latest cached version of the dataset since eZWALT/rlhf_reward_data_raw.json couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/walterjtv/.cache/huggingface/datasets/eZWALT___rlhf_reward_data_raw.json/default/0.0.0/42c2273ae129385c726ee00438e8d0992a822227 (last modified on Mon Oct 20 22:05:40 2025).


DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 1200
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 150
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 150
    })
})
Train / Val / Test sizes: 1200 150 150


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 24.34ba/s]
Processing Files (1 / 1): 100%|██████████|  805kB /  805kB,  503kB/s  
New Data Upload: 100%|██████████|  805kB /  805kB,  503kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 202.45ba/s]
Processing Files (1 / 1): 100%|██████████|  104kB /  104kB,  261kB/s  
New Data Upload: 100%|██████████|  104kB /  104kB,  261kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 195.62ba/s]
Processing Files (1 / 1): 100%|██████████|  112kB /  112kB,  280kB/s  
New Data Upload: 100%|██████████|  112kB /  112kB,  280kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.13s/ shards]


Pushed split dataset to hub at: eZWALT/rlhf_reward_splits_raw


In [12]:
model = AutoModelForSequenceClassification.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct", dtype=torch.bfloat16)
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

# Configuration for the Reward Model training loop
reward_config = RewardConfig(bf16=False)
# important to include the score head when base model is not a sequence classification model
lora_config = LoraConfig(modules_to_save=["score"])

reward_trainer = RewardTrainer(
    model=model,
    train_dataset=dataset,
    #eval_dataset=val_ds,
    args=reward_config,
    #peft_config=lora_config,
)

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at HuggingFaceTB/SmolLM2-135M-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ValueError: A processing_class must be specified when using the default RewardDataCollatorWithPadding

In [ ]:
reward_trainer.train()